# Predicting the Gender Wage Gap using Machine Learning!

## 📌 Goal of this notebook

In this notebook, we will:
- Explore a survey dataset about people working in tech
- Clean and prepare the data
- Train a Machine Learning model to predict **salary range (income bracket)**
- Investigate whether features like **gender, experience, and education** impact salary

⚠️ Important:
This is a simplified project for learning purposes. The results do **not** represent real-world fairness or causation.

📊 Data source:
[Kaggle](https://www.kaggle.com/code/paultimothymooney/2018-kaggle-machine-learning-data-science-survey/input?select=multipleChoiceResponses.csv)

In [ ]:
# optional: silence warnings for readability
import warnings

warnings.filterwarnings("ignore")

## Data 

### Import libraries

In [ ]:
# data
import pandas as pd

# plots
import matplotlib.pyplot as plt
import seaborn as sns

### Data import

In [ ]:
dataset = pd.read_csv("data/multipleChoiceResponses.csv")
dataset.head()

### Data cleaning

In [ ]:
# create a new var with the questions names
question_names = dataset.iloc[0]
question_names

In [ ]:
# dropping questions names flor clarity
dataset = dataset.drop(0, axis=0)
dataset.head()

### Data selection

In [ ]:
# print questions
question_names.head(10)

In [ ]:
# select data of interest
df_short = dataset[["Q1", "Q2", "Q3", "Q4", "Q5", "Q6", "Q7", "Q8", "Q9"]]
df_short = df_short.rename(
    columns={
        "Q1": "Gender",
        "Q2": "Age",
        "Q3": "Country",
        "Q4": "Education",
        "Q5": "FieldOfStudies",
        "Q6": "JobTitle",
        "Q7": "Industry",
        "Q8": "Experience",
        "Q9": "YearlyCompensation",
    }
)
df_short

In [ ]:
# print info
df_short.info()

### Treat missing values

In [ ]:
# fill NAs with a string (keeping rows, however not providing a lot of info)
df_short = df_short.fillna("Unknown")

In [ ]:
# observe the target values
df_short["YearlyCompensation"].value_counts()

In [ ]:
# drop the rows where the target is not available
df_short = df_short[
    (
        df_short["YearlyCompensation"]
        != "I do not wish to disclose my approximate yearly compensation"
    )
    & (df_short["YearlyCompensation"] != "Unknown")
]
df_short.head()

In [ ]:
# observe the target values
df_short["YearlyCompensation"].value_counts()

### Data exploration

In [ ]:
# working on a separate df for plots
df_plots = df_short.copy()

In [ ]:
# creating ordered categories for plotting
salary_order = [
    "0-10,000",
    "10-20,000",
    "20-30,000",
    "30-40,000",
    "40-50,000",
    "50-60,000",
    "60-70,000",
    "70-80,000",
    "80-90,000",
    "90-100,000",
    "100-125,000",
    "125-150,000",
    "150-200,000",
    "200-250,000",
    "250-300,000",
    "300-400,000",
    "400-500,000",
    "500,000+",
]
df_plots["YearlyCompensation"] = pd.Categorical(
    df_plots["YearlyCompensation"], categories=salary_order, ordered=True
)

In [ ]:
#### Salary vs Gender
sns.countplot(data=df_plots, x="YearlyCompensation", hue="Gender", order=salary_order)
plt.xticks(rotation=45)
plt.title("Salary distribution by gender")
plt.ylabel("Salary range")
plt.show()

In [ ]:
# simplify labels
country_map = {
    "United States of America": "USA",
    "United Kingdom of Great Britain and Northern Ireland": "UK",
}

df_plots["Country"] = df_plots["Country"].replace(country_map)

In [ ]:
df_plots["Country"].value_counts().head(10).plot(kind="bar")
plt.title("Top countries in dataset")
plt.show()

In [ ]:
# simplify labels
edu_map = {
    "Bachelor’s degree": "BSc",
    "Master’s degree": "MSc",
    "Doctoral degree": "PhD",
    "Professional degree": "Other degree",
    "Some college/university study without earning a bachelor’s degree": "Other degree",
    "No formal education past high school": "High School",
    "I prefer not to answer": "No Answer",
}

df_plots["Education_short"] = df_plots["Education"].replace(edu_map)

In [ ]:
# plot
plt.figure(figsize=(10, 5))
ax = sns.countplot(data=df_plots, x="Education_short", hue="YearlyCompensation")

plt.title("Education vs Salary")
ax.legend(title="Salary", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.xlabel("Education")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

### Data types

In [ ]:
# check data types
df_short.info()

#### Features

We convert ranges like "5-10 years" into a single number (7.5) so the model can use it. This is a sort of *feature engineering*.

In [ ]:
# Midpoint mapping for Experience
dic_exp = {
    "0-1": 0.5,
    "1-2": 1.5,
    "2-3": 2.5,
    "3-4": 3.5,
    "4-5": 4.5,
    "5-10": 7.5,
    "10-15": 12.5,
    "15-20": 17.5,
    "20-25": 22.5,
    "25-30": 27.5,
    "30 +": 30,
    "Unknown": 0,
}

# Apply mapping
df_short["Experience"] = df_short["Experience"].map(dic_exp)

In [ ]:
# plot num features distribution
sns.histplot(data=df_short, x="Experience");

In [ ]:
# Midpoint mapping for Age
dic_age = {
    "30-34": 32,
    "22-24": 23,
    "35-39": 37,
    "18-21": 19.5,
    "40-44": 42,
    "25-29": 27,
    "55-59": 57,
    "60-69": 64.5,
    "45-49": 47,
    "50-54": 52,
    "70-79": 74.5,
    "80+": 80,
}

# Apply mapping
df_short["Age"] = df_short["Age"].map(dic_age)

In [ ]:
# plot num features distribution
sns.histplot(data=df_short, x="Age");

In [ ]:
# check data types
df_short.info()

In [ ]:
# categorise features according to datatype
num_vars = ["Age", "Experience"]
cat_vars = [
    "Gender",
    "Country",
    "Education",
    "FieldOfStudies",
    "JobTitle",
    "Industry",
]

## Machine Learning

In [ ]:
from sklearn.model_selection import train_test_split

### Data preparation

#### Split features and target

In [ ]:
# define labels and features
labels_col = "YearlyCompensation"

X = df_short.drop(labels_col, axis=1)
y = df_short[labels_col]

In [ ]:
# display head of features X
X.head()

In [ ]:
# display head of target y
y.head()

#### Encode variables

Machines cannot understand text (like "Male" or "Engineer"), so we convert them into numbers.

In [ ]:
# replace objects by numerical categories
from sklearn.preprocessing import OrdinalEncoder

enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X[cat_vars] = enc.fit_transform(X[cat_vars])
X.head()

### The holdout method

We train the model on one part of the data and evaluate it on unseen data to simulate real-world performance.

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/machine-learning/train_test_split_basic.png" width="400">

In [ ]:
# split into Train/Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

### Fitting a ML model

In [ ]:
# import a model
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression()

In [ ]:
# fit the model
log_model.fit(X_train, y_train)

### Scoring the model performance

In [ ]:
# score the model on the Test data
log_model.score(X_test, y_test)

### Cross validation

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/machine-learning/K_fold_3.png" width="400">

In [ ]:
from sklearn.model_selection import cross_validate

# 5-Fold Cross validate model
cv_results = cross_validate(log_model, X, y, cv=5)

# scores
print(cv_results["test_score"])

In [ ]:
# mean of scores
scores_mean = float(cv_results["test_score"].mean())
print(f"CV scores mean using a logistic regression model is {scores_mean:.2f}")

**Cross-validation does not output a trained model, it only scores a hypothetical model trained on the entire dataset.**

## Iterate and improve

### Scale the numerical features

In [ ]:
# import a scaler
from sklearn.preprocessing import StandardScaler

# create a reusable variable
std_scaler = StandardScaler()

# create a X copy for safety
X_scaled = X.copy()

# fit the scaler
X_scaled[num_vars] = std_scaler.fit_transform(X[num_vars])

# split into Train/Test
X_train_scaled, X_test, y_train_scaled, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42
)

In [ ]:
# 5-Fold Cross validate model
cv_results_scaled = cross_validate(log_model, X_scaled, y, cv=5)

# mean of scores
scores_mean_scaled = float(cv_results_scaled["test_score"].mean())
print(
    f"CV scores mean using a scaled logistic regression model is {scores_mean_scaled:.2f}"
)

## Try a better model

In [ ]:
from lightgbm import LGBMClassifier
from lightgbm import plot_importance

In [ ]:
# create the model instance
lgbm_model = LGBMClassifier(verbosity=-1)  # adjusting verbosity to reduce the output

# fit the model
lgbm_model.fit(X_train_scaled, y_train)

# 5-Fold Cross validate model
cv_results_lgbm = cross_validate(lgbm_model, X_scaled, y, cv=5)

# mean of scores
scores_mean_lgbm = float(cv_results_lgbm["test_score"].mean())
print(f"CV scores mean using a scaled LGBM model is {scores_mean_lgbm:.2f}")

## Features importance

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

plot_importance(lgbm_model, max_num_features=50, height=0.8, ax=ax)
ax.grid(False)
plt.ylabel("Feature", size=12)
plt.xlabel("Importance", size=12)
plt.title("Features importance with LGBM model", fontsize=15)
plt.show();

## Predict new data

In [ ]:
# create a tech worker profile
Veronica = pd.DataFrame(
    {
        "Gender": ["Female"],
        "Age": [35],
        "Country": ["Italy"],
        "Education": ["Bachelor’s degree"],
        "FieldOfStudies": ["Computer science (software engineering, etc.)"],
        "JobTitle": ["Software Engineer"],
        "Industry": ["Computers/Technology"],
        "Experience": [10],
    }
)

# reuse our encoder to transform the data
Veronica[cat_vars] = enc.transform(Veronica[cat_vars])

# use our fitted model to estimate their yearly pay
Veronica_salary = lgbm_model.predict(Veronica)

# display result
print(f"Ahsvi's salary is estimated to be within the range of {Veronica_salary[0]} USD")

In [ ]:
# create a tech worker profile
John = pd.DataFrame(
    {
        "Gender": ["Male"],
        "Age": [52],
        "Country": ["Finland"],
        "Education": ["Master’s degree"],
        "FieldOfStudies": ["Computer science (software engineering, etc.)"],
        "JobTitle": ["Software Engineer"],
        "Industry": ["Computers/Technology"],
        "Experience": [20],
    }
)

# reuse our encoder to transform the data
John[cat_vars] = enc.transform(John[cat_vars])

# use our fitted model to estimate their yearly pay
John_salary = lgbm_model.predict(John)

# display result
print(f"John's salary is estimated to be within the range of {John_salary[0]} USD")

## Get more specific

What is the effect of gender on the wage gap in a specific country?

In [ ]:
def analyse_wage_gap_pipeline(
    df: pd.DataFrame, country: str, display_plots: bool = True
):
    """
    Process a dataset to analyse wage gap for a specific country using a LightGBM classifier.

    Steps:
    1. Filter data for the specified country.
    2. Encode categorical features and scale numerical features.
    3. Split data into training and test sets.
    4. Train a LightGBM classifier.
    5. Perform 5-fold cross-validation and report mean accuracy.
    6. Optionally plot feature importances.
    """

    # log
    print(f"Starting wage gap analysis for: {country}")

    # create a subset
    df_subset = df[df["Country"] == country]
    print(f"Subset created with {len(df_subset)} rows")

    # split features and target
    X = df_subset.drop(labels_col, axis=1)
    y = df_subset[labels_col]

    # encode categorical variables
    X[cat_vars] = enc.fit_transform(X[cat_vars])

    # scale numerical variables
    num_vars = X.columns.difference(cat_vars)
    X[num_vars] = std_scaler.fit_transform(X[num_vars])

    # split into Train/Test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42
    )
    print(f"Training on {len(X_train)} samples, testing on {len(X_test)} samples")

    # instantiate the model
    model = LGBMClassifier(verbosity=-1)

    # fit the model
    model.fit(X_train, y_train)

    # 5-Fold Cross validate model
    cv_results = cross_validate(model, X, y, cv=5, scoring="accuracy")

    # mean of scores
    scores_mean = float(cv_results["test_score"].mean())
    print(f"CV mean accuracy using LGBM model: {scores_mean:.2f}")

    if display_plots:
        # feature importance
        _, ax = plt.subplots(figsize=(10, 5))
        plot_importance(model, max_num_features=50, height=0.8, ax=ax)
        ax.grid(False)
        plt.ylabel("Feature", size=12)
        plt.xlabel("Importance", size=12)
        plt.title(f"Feature importance with LGBM model - {country}", fontsize=15)
        plt.show()

In [ ]:
analyse_wage_gap_pipeline(df=df_short, country="India")

In [ ]:
analyse_wage_gap_pipeline(df=df_short, country="United States of America")